# Session 1: arrows that add up
**A. Real-world problem.** You row a boat straight at a landing on the far bank of a river.
The water carries you sideways while you row. Where do you land? Two motions happen at once,
and each one is an *arrow* (a vector): how fast and which way.

**Plan.** (1) store an arrow as two numbers, (2) add arrows head to tail, (3) apply it to a
boat crossing a river, (4) find the heading that lands the boat straight across.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
WIDTH = 60.0     # river width in metres (the bank-to-bank distance)

## Step 1: an arrow is a pair of numbers
We use x for "downstream, along the river" and y for "across the river, to the far bank".
An arrow (vx, vy) says: move vx along the river and vy across, every second. Its *length*
is the speed, its *direction* is where it points.

The boat's own arrow (its velocity relative to the water) has speed `speed` and points
`heading` degrees away from straight across. Heading 0 is straight across; a positive
heading leans *upstream* (against the current, towards negative x).

In [ ]:
def boat_velocity(speed, heading_deg):
    """Arrow of the boat relative to the water, in m/s. x = downstream, y = across."""
    h = np.radians(heading_deg)
    return np.array([-speed * np.sin(h), speed * np.cos(h)])


def current_velocity(current):
    """Arrow of the river current: it points downstream (+x)."""
    return np.array([current, 0.0])


def length(arrow):
    """How long the arrow is (its speed)."""
    return float(np.hypot(arrow[0], arrow[1]))

## Step 2: adding arrows, head to tail
Put the tail of the second arrow on the head of the first. The sum is the arrow from the
very first tail to the very last head. In numbers you simply add the matching entries:
(a_x + b_x, a_y + b_y). Order does not matter: a + b = b + a.

In [ ]:
def add(a, b):
    return np.asarray(a, dtype=float) + np.asarray(b, dtype=float)


def ground_velocity(speed, heading_deg, current):
    """The boat's real motion over the ground = boat arrow + current arrow."""
    return add(boat_velocity(speed, heading_deg), current_velocity(current))

## Step 3: where does the boat land?
The boat crosses at the speed of the "across" part of its ground arrow. So the time is
width / vy, and the sideways drift is vx times that time.

In [ ]:
def crossing(speed, heading_deg, current, width=WIDTH):
    """Return (landing x in metres, crossing time in seconds). x > 0 means downstream."""
    g = ground_velocity(speed, heading_deg, current)
    if g[1] <= 1e-9:
        raise ValueError("the boat never reaches the far bank (no across motion)")
    t = width / g[1]
    return float(g[0] * t), float(t)


def heading_to_land_opposite(speed, current):
    """Heading (degrees upstream of straight across) that cancels the current.
    Returns None if the current is as fast as the boat or faster."""
    if current >= speed:
        return None
    return float(np.degrees(np.arcsin(current / speed)))

## Step 4: see it
Left: the arrows for a boat at 4 m/s in a 2 m/s current. Right: the paths over a 60 m
river for three headings.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    v, u = 4.0, 2.0
    fig, ax = plt.subplots(figsize=(5.5, 5))
    b = boat_velocity(v, 0.0)
    c = current_velocity(u)
    g = add(b, c)
    ax.annotate("", xy=b, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="tab:blue", lw=2.5))
    ax.annotate("", xy=g, xytext=b, arrowprops=dict(arrowstyle="->", color="tab:orange", lw=2.5))
    ax.annotate("", xy=g, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="k", lw=2.5))
    ax.text(-1.2, 2.0, "boat\n4 m/s", color="tab:blue")
    ax.text(0.2, 4.25, "current 2 m/s", color="tab:orange")
    ax.text(1.3, 1.7, f"ground velocity\n{length(g):.2f} m/s", color="k")
    ax.set_xlim(-1.5, 3.5)
    ax.set_ylim(-0.5, 5)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    ax.set_xlabel("x: downstream (m/s)")
    ax.set_ylabel("y: across the river (m/s)")
    ax.set_title("Head to tail: boat arrow + current arrow")
    fig.tight_layout()
    fig.savefig(OUT / "head_to_tail.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6, 4.5))
    for h, col in [(0, "tab:red"), (15, "tab:orange"), (heading_to_land_opposite(v, u), "tab:green")]:
        x_land, t = crossing(v, h, u)
        ax2.plot([0, x_land], [0, WIDTH], color=col, lw=2.5,
                 label=f"heading {h:.0f} deg: lands {x_land:.1f} m, {t:.1f} s")
        print(f"heading {h:5.1f} deg -> lands {x_land:6.2f} m downstream after {t:5.2f} s")
    ax2.axhline(0, color="k")
    ax2.axhline(WIDTH, color="k")
    ax2.set_xlabel("distance downstream (m)")
    ax2.set_ylabel("distance across the river (m)")
    ax2.set_title("Boat 4 m/s, current 2 m/s, river 60 m")
    ax2.legend(fontsize=8, loc="lower right")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "river_paths.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change `crossing(4.0, 0.0, 2.0)` to a 5 m/s boat and a 3 m/s current and predict the time and drift before you run it. Compare with Section F, question B.
2. **Stretch.** Write `head_to_tail(a, b)` that returns the three points tail, joint and head of the two arrows a and b, and use it to redraw the figure for a boat of 3 m/s and a current of 1 m/s.
3. **Explain.** Explain why a current that is faster than the boat makes it impossible to land opposite, using the idea that the along-river part of the ground arrow must be zero.

## Assessment
- **A. Conceptual.** Why must the boat point upstream to land exactly opposite its start? What does the sideways part of the boat arrow have to do?
- **B. Numerical.** A boat of 5 m/s points straight across a 60 m river with a 3 m/s current. Find the crossing time, the landing point and the speed over the ground.
- **C. Graph and simulation interpretation.** On the landing-point graph, for a 4 m/s boat and a 2 m/s current, where does the curve cross zero and what is the landing point at heading 0? What does the curve do for negative headings (leaning downstream)?
- **D. Design and optimisation challenge.** Design: the pier is 15 m downstream of the point opposite the start. Choose a heading for the 4 m/s boat in the 2 m/s current so that it lands at the pier, and state how long the crossing takes.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    x, t = L.crossing(4.0, 0.0, 2.0)
    checks.append(("worked example: boat 4 m/s straight across, current 2 m/s, 60 m: lands 30 m downstream after 15 s",
                   abs(x - 30.0) < 1e-9 and abs(t - 15.0) < 1e-9))
    a = L.add([3.0, 0.0], [0.0, 4.0])
    checks.append(("(3, 0) + (0, 4) = (3, 4) with length 5", np.allclose(a, [3.0, 4.0]) and abs(L.length(a) - 5.0) < 1e-12))
    p, q = np.array([1.0, 2.0]), np.array([-3.0, 0.5])
    checks.append(("a + b = b + a", np.allclose(L.add(p, q), L.add(q, p))))
    h = L.heading_to_land_opposite(4.0, 2.0)
    checks.append(("heading 30 degrees upstream lands exactly opposite", abs(h - 30.0) < 1e-9 and abs(L.crossing(4.0, h, 2.0)[0]) < 1e-9))
    x30, t30 = L.crossing(4.0, 30.0, 2.0)
    checks.append(("that crossing takes 17.32 s (= 60 / (4 cos 30))", abs(t30 - 60.0 / (4.0 * np.cos(np.radians(30)))) < 1e-9 and abs(t30 - 17.32) < 0.01))
    checks.append(("boat arrow keeps the boat speed for every heading",
                   all(abs(L.length(L.boat_velocity(4.0, d)) - 4.0) < 1e-12 for d in range(-60, 61, 15))))
    checks.append(("no current: straight heading lands straight across", abs(L.crossing(4.0, 0.0, 0.0)[0]) < 1e-12))
    checks.append(("current equal to boat speed: no heading cancels it", L.heading_to_land_opposite(2.0, 2.0) is None))
    try:
        L.crossing(4.0, 90.0, 2.0)
        ok = False
    except ValueError:
        ok = True
    checks.append(("heading 90 degrees (along the bank) never crosses: error raised", ok))
    x15, t15 = L.crossing(4.0, 15.0, 2.0)
    checks.append(("design answer: heading 15 degrees lands about 15 m downstream (14.98 m) after 15.53 s", abs(x15 - 14.98) < 0.01 and abs(t15 - 15.53) < 0.01))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
